# Residual trajectories

Follow the residual stream through recurrent layers in two-dimensional PCA space.
The shared main-paper models use 32 recurrences with one fixed text and three
seeded GSM8K examples. The main-paper Parcae model is 770M.
The two randomly initialised models use one seeded sequence of random token IDs.

Both modes compute the same examples and figures. `mode="quick"` saves only in the
run directory; `mode="full"` also updates the paper figures. The GSM8K grid always
uses three examples, independently of mode. SLURM selects full mode.

In [ ]:
mode = "quick"  # "full" also updates paper figures; computation is identical.
recurrences = 32  # All pretrained main-paper models.
model_recurrences = None  # Defaults to the shared main group; optionally list pairs.
random_model_recurrences = [
    ("relu_no_injection", 64),
    ("gated_input_injection", 64),
]
sequence = "Hello! I've been well. I hope that you're doing well."
token_position = -1
plot_recurrences = 32
gsm8k_token_fractions = [0.0, 0.5, 1.0]  # First, middle, final question tokens.
random_width = 512
random_heads = 4
random_sequence_length = 32
seed = 42
device = "auto"
dtype = "auto"  # Pretrained precision; random models use float32.
output_dir = None

cache_dir = None  # Defaults to LOOPED_INTERNALS_CACHE_DIR or /data-gauss/hugbla/looped-internals-cache.
cache_mode = None  # Environment default; "reuse", "require" (no inference), or "off".
analysis_version = 1  # Bump when this notebook's scientific reduction changes.
source_run = None  # Optional completed run directory with matching scientific settings.

In [ ]:
import sys
from pathlib import Path

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
import gc
from dataclasses import asdict
from importlib.metadata import version

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.decomposition import PCA

from experiments.cache import ExperimentCache
from experiments.paper_models import PAPER_MODEL_GROUPS
from experiments.saved_runs import SavedRun
from experiments.support import (
    COLORMAPS,
    MODEL_LABELS,
    _set_seed,
    add_depth_colorbar,
    configure_experiment,
    export_paper_figures,
    get_figsize,
    load_experiment_model,
    sample_indices,
    save_figure,
    save_results,
    write_tokenization,
)
from looped_internals.capture import compute_model_run_data
from looped_internals.evaluation.lm_eval import BridgeLM, get_task_manager
from looped_internals.models import MinimalRecurrentConfig, ModelSpec, load_model
from looped_internals.models.tokenization import tokenizer_metadata

plt.style.use(ROOT / "plots" / "matlab.mplstyle")

# Resolve defaults after parameter overrides, as in the other paper notebooks.
if model_recurrences is None:
    model_recurrences = [(name, recurrences) for name in PAPER_MODEL_GROUPS["main"]]
model_recurrences = [tuple(pair) for pair in model_recurrences]
random_model_recurrences = [tuple(pair) for pair in random_model_recurrences]
configs = {
    (name, recurrences): configure_experiment(
        experiment="trajectories",
        mode=mode,
        models=[name],
        num_steps=recurrences,
        limit=1,
        seed=seed,
        device=device,
        dtype=dtype,
        output_dir=output_dir,
    )
    for name, recurrences in model_recurrences
}
config = next(iter(configs.values()))

result_cache = ExperimentCache(cache_dir, cache_mode)

In [ ]:
# Both random models use Huginn-style normalization.
random_steps = dict(random_model_recurrences)
RANDOM_MODELS = {
    "relu_no_injection": dict(
        n_layers=24, input_injection=None, gated_mlp=False, activation="relu"
    ),
    "gated_input_injection": dict(
        n_layers=12, input_injection="projection", gated_mlp=True
    ),
}
random_configs = {
    (name, recurrences): MinimalRecurrentConfig(
        d_model=random_width,
        n_heads=random_heads,
        num_cycles=recurrences,
        norm_pre_attn=True,
        norm_post_attn=False,
        norm_pre_mlp=True,
        norm_post_mlp=False,
        norm_post_attn_residual=True,
        norm_post_mlp_residual=True,
        norm_post_cycle=True,
        **RANDOM_MODELS[name],
    )
    for name, recurrences in random_model_recurrences
}


def summarize_trajectory(run, position):
    """Fit PCA to the selected token, then group recurrent coordinates by layer."""
    blocks = run.per_virtual_layer_metrics
    residuals = torch.stack(
        [block.raw_residual[0, position].float() for block in blocks]
    ).numpy()
    pca = PCA(n_components=2, svd_solver="full")
    points = pca.fit_transform(residuals)
    layers = run.recurrent_layer_indices
    indices = [
        [block.virtual_layer_index for block in blocks if block.layer_index == layer]
        for layer in layers
    ]
    # Shape: (recurrences, recurrent layers, 2). PCA also sees prelude/coda states.
    coordinates = np.stack([points[index] for index in indices], axis=1)
    return {
        "coordinates": coordinates,
        "layers": layers,
        "input_ids": run.input_ids.tolist(),
        "token_position": position % run.input_ids.shape[-1],
        "explained_variance_ratio": pca.explained_variance_ratio_.tolist(),
    }


@torch.no_grad()
def capture_trajectories(bridge, structure, tokens, positions):
    _, cache = bridge.run_with_cache(
        tokens, accumulate=True, names_filter=structure.hook_names(), device="cpu"
    )
    run = compute_model_run_data(
        cache, tokens, structure, compute_pairwise=False, store_raw_values=True
    )[0]
    return [summarize_trajectory(run, position) for position in positions]


def gsm8k_inputs(bridge, example):
    """Match likelihood-task inputs and locate tokens inside the question."""
    context, answer = example["context"], example["answer"]
    context_ids, answer_ids = BridgeLM(bridge, compute_metrics=False)._encode_pair(
        context, answer
    )
    tokens = torch.tensor([context_ids + answer_ids], device=config.device)
    # Native offsets exclude BOS and the Question:/Answer: wrapper from selection.
    offsets = bridge.tokenizer(context, return_offsets_mapping=True)["offset_mapping"]
    start = context.index(example["question"])
    end = start + len(example["question"])
    question_positions = [
        i for i, (left, right) in enumerate(offsets) if right > start and left < end
    ]
    positions = [
        question_positions[round(fraction * (len(question_positions) - 1))]
        for fraction in gsm8k_token_fractions
    ]
    return tokens, positions

## Generate and reduce

Load each model–recurrence pair once, caching only residual outputs. Each
pretrained model processes the fixed text and the same three GSM8K examples.
Reduce all selected tokens after each forward pass, then release its cache.
The saved `overview.pt` contains coordinates, layer identities, input IDs,
selected positions, decoded tokens, and explained variance; plotting can then be
repeated without inference.

GSM8K uses the first three test indices from the same seeded shuffle as the other
experiments, in shuffle order (588, 1214, 57 with `seed=42`). Inputs use the
`gsm8k_likelihood` task's question/answer format, including the reference answer,
and each checkpoint's native tokenization. The grid selects tokens within the
question, excluding special tokens and the surrounding prompt. Token boundaries
and absolute positions can differ between models.

Each random model is initialised independently with the chosen seed. Both receive
the same random token IDs. These are individual illustrative examples, not averages
across initialisations.

In [ ]:
task = get_task_manager().load(["gsm8k_likelihood"])["tasks"]["gsm8k_likelihood"]
gsm8k_indices = sample_indices(len(task.eval_docs), 3, seed)
gsm8k_examples = []
for index in gsm8k_indices:
    doc = task.eval_docs[index]
    request = task.construct_requests(
        doc, ctx=task.doc_to_text(doc), metadata=(task.config.task, index, 1)
    )
    context, answer = request.args
    gsm8k_examples.append(
        dict(index=index, question=doc["question"], context=context, answer=answer)
    )
print(f"GSM8K test indices: {gsm8k_indices}")

In [ ]:
settings = asdict(config)
for field in ("models", "num_steps", "task", "limit"):
    del settings[field]
settings.update(
    dataset="fixed_sequence",
    output_dir=str(config.output_dir),
    model_recurrences=model_recurrences,
    random_model_recurrences=random_model_recurrences,
    random_model_configs={
        name: asdict(value) for (name, _), value in random_configs.items()
    },
    random_dtype="float32",
    random_sequence_length=random_sequence_length,
    sequence=sequence,
    token_position=token_position,
    plot_recurrences=plot_recurrences,
    gsm8k=dict(
        task="gsm8k_likelihood",
        dataset="openai/gsm8k",
        subset="main",
        split="test",
        indices=gsm8k_indices,
        token_fractions=gsm8k_token_fractions,
        token_region="question",
    ),
)
settings["versions"] = {
    name: version(name)
    for name in ("torch", "transformer-lens", "transformers", "scikit-learn")
}
settings["cache"] = result_cache.settings
previous = SavedRun(source_run, settings, analysis_version=analysis_version)
previous.check_pairs(settings)
settings.update(analysis_version=analysis_version, source_run=source_run)
run_dir = save_results({}, settings, config.output_dir)

summaries = {}
gsm8k_summaries = {}


def collect_pretrained_trajectory(name, pair_config):
    _set_seed(seed)
    bridge, structure = load_experiment_model(
        name, pair_config, **({"cache": result_cache} if result_cache.enabled else {})
    )
    tokens = bridge.tokenizer.encode(sequence, return_tensors="pt").to(config.device)
    (summary,) = capture_trajectories(bridge, structure, tokens, [token_position])
    gsm8k_summary = []
    for example in gsm8k_examples:
        tokens, positions = gsm8k_inputs(bridge, example)
        trajectories = capture_trajectories(bridge, structure, tokens, positions)
        for trajectory in trajectories:
            position = trajectory["token_position"]
            trajectory["token_text"] = bridge.tokenizer.decode(
                [tokens[0, position].item()]
            )
        gsm8k_summary.append(dict(**example, trajectories=trajectories))
    tokenization = {"bos_policy": "native", **tokenizer_metadata(bridge.tokenizer)}
    del bridge, tokens
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return summary, gsm8k_summary, tokenization


for pair, pair_config in configs.items():
    if pair in previous:
        summaries[pair] = previous.data["summaries"][pair]
        gsm8k_summaries[pair] = previous.data["gsm8k"][pair]
        write_tokenization(run_dir, pair[0], previous.tokenization(pair[0]))
        continue
    name, recurrences = pair
    key = result_cache.key(
        pair_config,
        name,
        analysis_version=analysis_version,
        sequence=sequence,
        token_position=token_position,
        gsm8k_examples=gsm8k_examples,
        gsm8k_token_fractions=gsm8k_token_fractions,
    )
    summaries[pair], gsm8k_summaries[pair], tokenization = result_cache.run(
        "trajectories", key, collect_pretrained_trajectory, name, pair_config
    )
    write_tokenization(run_dir, name, tokenization)


def collect_random_trajectory(model_config):
    _set_seed(seed)
    bridge, structure = load_model(
        ModelSpec(model_config), device=config.device, dtype=torch.float32
    )
    # An independent generator keeps the input fixed across the two architectures.
    generator = torch.Generator().manual_seed(seed)
    tokens = torch.randint(
        model_config.d_vocab, (1, random_sequence_length), generator=generator
    ).to(config.device)
    (summary,) = capture_trajectories(bridge, structure, tokens, [token_position])
    del bridge, tokens
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return summary


for pair, model_config in random_configs.items():
    if pair in previous:
        summaries[pair] = previous.data["summaries"][pair]
        continue
    key = result_cache.key(
        config,
        analysis_version=analysis_version,
        model_config=asdict(model_config),
        sequence_length=random_sequence_length,
        token_position=token_position,
    )
    summaries[pair] = result_cache.run(
        "random_trajectories", key, collect_random_trajectory, model_config
    )

torch.save(
    {"summaries": summaries, "gsm8k": gsm8k_summaries, "run": settings},
    run_dir / "overview.pt",
)
result_cache.write_manifest(run_dir)

## Reading a trajectory

| Element | Meaning |
| --- | --- |
| A point | The selected token's residual stream after one recurrent layer |
| A line | Successive layers within one recurrence, in execution order |
| Colour | Early to late recurrence within the panel |
| Panels | Consecutive portions of the displayed rollout, with shared axes |
| PC 1 and PC 2 | The two directions of greatest variance for this token's residuals |

Fit PCA **separately for each model and selected token**, using every virtual layer
in its complete rollout, including prelude and coda where present. Then display
only recurrent layers from the first `plot_recurrences` recurrences. The random
models therefore fit their axes using all 64 recurrences while showing the first
32. The panels for each model/sequence/token combination share a PCA basis
and axis limits; different combinations have independent bases.

The default token is the final token. Lines connect layers within a recurrence;
they do not join the last layer to the first layer of the next recurrence. A loop
in this projection illustrates the within-recurrence path through layers, and does
not by itself establish an orbit across recurrences. The random examples differ in
layer count and MLP architecture as well as injection, so they are illustrative
rather than a controlled comparison of injection alone.

In [ ]:
def draw_trajectory(axes, points):
    """Draw one token with a common PCA basis and limits across panels."""
    cmap = plt.colormaps[COLORMAPS["recurrences"]]
    lower = points.min(axis=(0, 1))
    upper = points.max(axis=(0, 1))
    padding = np.maximum(0.1 * (upper - lower), 1e-6)
    for ax, indices in zip(
        axes, np.array_split(np.arange(len(points)), len(axes)), strict=True
    ):
        colors = cmap(np.linspace(0, 1, len(indices)))
        for recurrence, color in zip(indices, colors, strict=True):
            path = points[recurrence]
            ax.plot(path[:, 0], path[:, 1], linewidth=2, color=color)
        ax.set(
            title=f"Recurrences {indices[0] + 1}-{indices[-1] + 1}",
            xlabel="PC 1",
            xlim=(lower[0] - padding[0], upper[0] + padding[0]),
            ylim=(lower[1] - padding[1], upper[1] + padding[1]),
        )
    axes[0].set_ylabel("PC 2")


def plot_trajectory(pair, panels=2, height_per_subplot=1.5, filename=None):
    """Show successive portions of one rollout in a shared two-dimensional PCA basis."""
    points = summaries[pair]["coordinates"][:plot_recurrences]
    if len(points) < panels:
        raise ValueError("Each panel needs at least one recurrence")
    fig, axes = plt.subplots(
        1,
        panels,
        squeeze=False,
        figsize=get_figsize(
            fullpage=panels > 2,
            subplots=(1, panels),
            height_per_subplot=height_per_subplot,
        ),
        layout="constrained",
    )
    draw_trajectory(axes[0], points)
    # Keep the endpoint labels clear of the right panel on a compact canvas.
    colorbar_pad = 0.05 if panels == 2 else 0.025
    add_depth_colorbar(fig, axes, "recurrence", pad=colorbar_pad)
    if filename is None:
        name, recurrences = pair
        suffix = "_extended" if panels == 4 else ""
        filename = f"pca_quartiles_{MODEL_LABELS[name]} ({recurrences} steps){suffix}"
    save_figure(fig, run_dir, filename, bbox_inches=None)

## Pretrained models: single-column figures

Two panels show recurrences 1–16 and 17–32 for each model. The fixed text is
encoded with each checkpoint's native tokenizer.

In [ ]:
for pair in model_recurrences:
    plot_trajectory(pair)

## Pretrained models: full-width figures

Four panels show eight recurrences each, using the same saved coordinates.

In [ ]:
for pair in model_recurrences:
    plot_trajectory(pair, panels=4)

## Random model: ReLU without input injection

A 24-layer model with an ungated ReLU MLP and Huginn-style normalization. This
single-column illustration uses a 1.5-inch-high canvas.

In [ ]:
plot_trajectory(
    ("relu_no_injection", random_steps["relu_no_injection"]),
    filename="splash_trajectory",
)

## Random model: gated MLP with projected input injection

A 12-layer model with a gated SiLU MLP and Huginn-style normalization. The
single-column canvas is two inches high.

In [ ]:
plot_trajectory(
    ("gated_input_injection", random_steps["gated_input_injection"]),
    height_per_subplot=2,
    filename="initialised_injected_trajectory",
)

## Pretrained models: GSM8K sequence and token grid

Each model has one full-width figure. Rows follow the three sampled sequences;
columns select the first, middle, and final question tokens by default. Each grid
cell contains two axes: **recurrences 1–16** and **17–32**. Its title records the
fraction through the question, the zero-based position in the full input, and the
decoded token. The leftmost labels identify the GSM8K test index.

Fit PCA separately for each token and sequence, then share its basis and limits
across the two recurrence panels. Colours restart from early to late within each
panel, as in the figures above. The grids show individual trajectories without
averaging across sequences or token positions.

In [ ]:
def plot_gsm8k_grid(pair):
    examples = gsm8k_summaries[pair]
    columns = len(gsm8k_token_fractions)
    fig = plt.figure(
        figsize=get_figsize(
            fullpage=True, subplots=(len(examples), columns), height_per_subplot=1.8
        ),
        layout="constrained",
    )
    # Reserve a right strip for one colour bar shared by all nine cells.
    fig.get_layout_engine().set(rect=(0, 0, 0.92, 1))
    fig.suptitle(MODEL_LABELS[pair[0]])
    cells = fig.subfigures(len(examples), columns, squeeze=False)
    for row, example in enumerate(examples):
        for column, trajectory in enumerate(example["trajectories"]):
            cell = cells[row, column]
            axes = cell.subplots(1, 2, sharex=True, sharey=True)
            points = trajectory["coordinates"][:plot_recurrences]
            draw_trajectory(axes, points)
            fraction = gsm8k_token_fractions[column]
            position = trajectory["token_position"]
            token = trajectory["token_text"]
            cell.suptitle(f"Question {fraction:.0%} · position {position}\n{token!r}")
            if column == 0:
                axes[0].set_ylabel(f"GSM8K #{example['index']}\nPC 2")
            for ax in axes:
                ax.locator_params(axis="both", nbins=3)
    add_depth_colorbar(
        fig,
        fig.axes,
        "recurrence",
        cax=fig.add_axes([0.955, 0.18, 0.006, 0.64]),
        label="Recurrence within each panel",
    )
    filename = f"pca_gsm8k_grid_{MODEL_LABELS[pair[0]]} ({pair[1]} steps)"
    save_figure(fig, run_dir, filename, bbox_inches=None)

In [ ]:
for pair in model_recurrences:
    plot_gsm8k_grid(pair)

In [ ]:
print(run_dir)
paper_dir = export_paper_figures(config, run_dir)
if paper_dir is not None:
    print(f"Saved paper figures to {paper_dir}")